# 01: Prompt Caching Economics, Context AST Budgeting & Retrieval Fidelity

> **Curriculum Phase**: `Phase 01 - Prompt and Context Engineering`  
> **Prerequisites**: Python 3.12+, Phase 00 fundamentals  
> **Key Concepts**: Prompt & Prefix Caching Economics, The Contiguous Prefix Invariant, Needle-In-A-Haystack (NIAH) Simulation, The Lost-in-the-Middle U-Curve, and Strongly-Typed Context AST Token Budgeting.  

Welcome to the companion notebook for **Phase 01: Prompt & Context Engineering**. In production AI architectures, treats the context window not as an untyped string buffer, but as a compiled Abstract Syntax Tree (AST) subject to rigorous systems constraints:

1. **Prompt Caching Break-Even Economics**: Quantifying dollar costs and TTFT latency across cold prefill misses vs cached prefix hits, and measuring the devastating financial penalty of 'Prefix Taint'.
2. **Needle-In-A-Haystack (NIAH) & Attention Dispersion**: Simulating the empirical Lost-in-the-Middle U-curve to visualize why advertised context sizes fail without boundary pinning and operational ceilings.
3. **Context AST Token Budget Portfolio**: Engineering a Pydantic v2 Context AST compiler with priority-based progressive compaction to guarantee deterministic token budgets.

In [ ]:
# %% [Setup] Colab & Environment Bootstrap
import os, sys
if 'google.colab' in str(get_ipython()):
    print('Running in Google Colab. Cloning repository...')
    !git clone --depth 1 https://github.com/dip4k/senior-engineer-to-ai-engineer-roadmap.git repo
    %cd repo
    !pip install -q -r agent-forge/requirements.txt -r notebooks/requirements-notebooks.txt
    sys.path.insert(0, os.path.abspath('agent-forge'))
else:
    # Local environment setup: locate repo root and agent-forge
    repo_root = os.path.abspath('..') if os.path.exists('../agent-forge') else os.path.abspath('.')
    af_path = os.path.join(repo_root, 'agent-forge')
    if os.path.exists(af_path) and af_path not in sys.path:
        sys.path.insert(0, af_path)
    print(f'Running locally with agent-forge in path: {af_path}')

### Environment Setup & Dependencies

We import standard libraries including Pydantic v2 for structured Context AST nodes, Matplotlib, and Seaborn.

In [ ]:
import math
from typing import Dict, List, Optional
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from pydantic import BaseModel, Field

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.dpi'] = 120
print('Phase 01 dependencies ready.')

## Part 1: Prompt Caching Break-Even Economics & The Prefix Invariant

Prompt caching (prefix caching) memoizes the intermediate Key and Value activation vectors of immutable prompt prefixes directly in GPU High-Bandwidth Memory (HBM). When consecutive requests share an identical token prefix, the serving engine skips redundant matrix math.

### The Contiguous Prefix Invariant & Prefix Taint
Self-attention uses positional encodings that start at **Token Index 0**. If even a single character or token changes at Token 0 (for example, inserting a dynamic timestamp or request UUID), every subsequent positional encoding changes. This causes **Prefix Taint**, resulting in a 0% cache hit rate and forcing full quadratic recomputation.

### Economics & Cost Structure
Major providers (Anthropic, OpenAI, Gemini) offer dramatic discounts for cached prefix tokens:
- **Standard Input Prefill**: e.g. 3.00 USD per million tokens (MTok).
- **Cache Write (Creation)**: e.g. 3.75 USD per MTok (small 25% initial premium).
- **Cache Read (Hit)**: e.g. 0.30 USD per MTok (**90% discount**).
- **Latency**: TTFT drops from ~1,800ms down to ~180ms on cache hits.

In [ ]:
class PromptCacheSimulator:
    """Simulates cost and latency across cold misses, cached hits, and tainted prefixes."""

    def __init__(
        self,
        prefix_tokens: int = 15_000,
        dynamic_tokens: int = 600,
        output_tokens: int = 300,
        cost_input_per_m: float = 3.00,       # $3.00 / MTok
        cost_cache_write_per_m: float = 3.75, # $3.75 / MTok
        cost_cache_read_per_m: float = 0.30,  # $0.30 / MTok (90% discount)
        cost_output_per_m: float = 15.00,     # $15.00 / MTok
        cold_ttft_ms: float = 1800.0,         # 1.8s cold TTFT
        warm_ttft_ms: float = 220.0,          # 220ms cached TTFT
    ):
        self.prefix_tokens = prefix_tokens
        self.dynamic_tokens = dynamic_tokens
        self.output_tokens = output_tokens
        self.cost_input_per_m = cost_input_per_m
        self.cost_cache_write_per_m = cost_cache_write_per_m
        self.cost_cache_read_per_m = cost_cache_read_per_m
        self.cost_output_per_m = cost_output_per_m
        self.cold_ttft_ms = cold_ttft_ms
        self.warm_ttft_ms = warm_ttft_ms

    def simulate(self, request_counts: List[int]) -> pd.DataFrame:
        records = []
        for N in request_counts:
            # 1. Uncached Baseline (100% cold prefill on all requests)
            uncached_input_cost = N * ((self.prefix_tokens + self.dynamic_tokens) / 1e6) * self.cost_input_per_m
            uncached_output_cost = N * (self.output_tokens / 1e6) * self.cost_output_per_m
            uncached_cost = uncached_input_cost + uncached_output_cost
            uncached_cum_ttft_sec = (N * self.cold_ttft_ms) / 1000.0

            # 2. Clean Cached Prefix (1 write, N-1 hits)
            cache_write_cost = 1 * (self.prefix_tokens / 1e6) * self.cost_cache_write_per_m
            cache_read_cost = max(0, N - 1) * (self.prefix_tokens / 1e6) * self.cost_cache_read_per_m
            cached_dynamic_cost = N * (self.dynamic_tokens / 1e6) * self.cost_input_per_m
            cached_cost = cache_write_cost + cache_read_cost + cached_dynamic_cost + uncached_output_cost
            cached_cum_ttft_sec = (1 * self.cold_ttft_ms + max(0, N - 1) * self.warm_ttft_ms) / 1000.0

            # 3. Tainted Prefix (e.g. Timestamp at Token 0 -> 0% hit rate, pays cold prefill every time)
            tainted_cost = uncached_cost
            tainted_cum_ttft_sec = uncached_cum_ttft_sec

            records.append({
                "Requests": N,
                "Uncached Cost (USD)": round(uncached_cost, 2),
                "Cached Cost (USD)": round(cached_cost, 2),
                "Tainted Cost (USD)": round(tainted_cost, 2),
                "Savings (%)": round(((uncached_cost - cached_cost) / uncached_cost) * 100, 1) if uncached_cost > 0 else 0,
                "Uncached TTFT (s)": round(uncached_cum_ttft_sec, 1),
                "Cached TTFT (s)": round(cached_cum_ttft_sec, 1),
            })
        return pd.DataFrame(records)

sim = PromptCacheSimulator(prefix_tokens=15_000, dynamic_tokens=500, output_tokens=250)
request_volume = [1, 2, 5, 10, 50, 100, 500, 1000, 2500]
results_df = sim.simulate(request_volume)
display(results_df)

In [ ]:
# Visualizing Prompt Caching Break-Even Economics
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Left Plot: Cumulative Cost
ax1.plot(results_df["Requests"], results_df["Uncached Cost (USD)"], 'r--', linewidth=2, label='Uncached / Tainted (0% Cache Hit)')
ax1.plot(results_df["Requests"], results_df["Cached Cost (USD)"], 'g-', linewidth=2.5, label='Clean Cached Prefix (90% Hit Discount)')
ax1.set_xlabel('Total Request Volume within Cache TTL Window')
ax1.set_ylabel('Cumulative API Cost (USD)')
ax1.set_title('Cumulative Cost Divergence (15,000-Token Static Prefix)', fontsize=11, pad=10)
ax1.legend(loc='upper left')
ax1.grid(True, linestyle='--', alpha=0.6)

# Right Plot: Cumulative Latency (TTFT)
ax2.plot(results_df["Requests"], results_df["Uncached TTFT (s)"], 'r--', linewidth=2, label='Uncached TTFT (1.8s per request)')
ax2.plot(results_df["Requests"], results_df["Cached TTFT (s)"], 'b-', linewidth=2.5, label='Cached TTFT (220ms per request)')
ax2.set_xlabel('Total Request Volume')
ax2.set_ylabel('Total Cumulative Wait Time (Seconds)')
ax2.set_title('Cumulative Time-To-First-Token (TTFT) Latency', fontsize=11, pad=10)
ax2.legend(loc='upper left')
ax2.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()

# Print exact break-even confirmation
break_even_n = 2  # Request 2 immediately pays off the small 25% write premium
print(f"Break-even is achieved at request {break_even_n}. At 2,500 requests, net cost reduction is {results_df.iloc[-1]['Savings (%)']}%")

## Part 2: Needle-In-A-Haystack (NIAH) Simulation & The Attention U-Curve

Marketing materials frequently boast 128K, 1M, or 2M token context windows. However, in production engineering, advertised capacity represents physical GPU memory limits, **not reasoning fidelity**.

### The Lost-in-the-Middle U-Curve (Liu et al., 2023)
Empirical transformer attention does not distribute uniformly across long contexts. Instead, retrieval and multi-hop reasoning follow a distinct U-curve:
- **Primacy Anchor (0%–10% Depth)**: Highest retrieval fidelity (88%–94%). Tokens near index 0 establish the initial attention reference frame.
- **The Middle Void (20%–80% Depth)**: Massive degradation. Information placed between 30% and 70% depth experiences a nadir dropping as low as 28%–45% accuracy.
- **Recency Anchor (90%–100% Depth)**: Retrieval rebounds to 85%–96% immediately preceding the generation tokens.

### The 50% Operational Ceiling Rule
Because attention dispersion accelerates as context expands, senior AI engineers enforce the **50% Operational Ceiling Rule**: never fill more than 50% of a model's rated context window with uncompacted documents.

In [ ]:
# Simulating empirical retrieval accuracy across Context Length vs Needle Depth
context_lengths = ["4K", "8K", "16K", "32K", "64K", "128K"]
depth_buckets = [
    "0-10% (Start)", "10-20%", "20-30%", "30-40%",
    "40-50% (Center)", "50-60%", "60-70%", "70-80%",
    "80-90%", "90-100% (End)"
]

# Empirical baseline U-curve values at 4K context
base_u_curve = np.array([0.96, 0.90, 0.74, 0.58, 0.48, 0.52, 0.65, 0.78, 0.89, 0.97])

# Degradation factors across context window lengths
length_penalties = [1.0, 0.96, 0.88, 0.78, 0.62, 0.45]

heatmap_data = np.zeros((len(depth_buckets), len(context_lengths)))

for col_idx, penalty in enumerate(length_penalties):
    for row_idx, base_val in enumerate(base_u_curve):
        # Middle depths (indices 3, 4, 5) experience stronger non-linear degradation
        is_middle = 3 <= row_idx <= 5
        middle_drag = 0.82 if is_middle else 1.0
        val = base_val * penalty * middle_drag
        # Clip to realistic bounds
        heatmap_data[row_idx, col_idx] = max(0.18, min(1.0, val))

df_heatmap = pd.DataFrame(heatmap_data * 100.0, index=depth_buckets, columns=context_lengths)

# Plotting the NIAH Attention U-Curve Heatmap
fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(
    df_heatmap, annot=True, fmt=".0f", cmap="RdYlGn", vmin=20, vmax=100,
    cbar_kws={'label': 'Retrieval / Reasoning Accuracy (%)'}, ax=ax, linewidths=0.5
)
ax.set_title('Needle-In-A-Haystack (NIAH) Empirical Fidelity: The Lost-in-the-Middle U-Curve', fontsize=12, pad=12)
ax.set_xlabel('Total Context Length', fontsize=11)
ax.set_ylabel('Needle Insertion Depth in Prompt Context', fontsize=11)
plt.tight_layout()
plt.show()

## Part 3: Context AST Architecture & Dynamic Token Budgeting Portfolio

To protect against delimiter collision injection, KV-cache thrashing, and context rot, production systems compile prompts as a **Context Abstract Syntax Tree (AST)** consisting of three strictly segregated tiers:

```text
┌─────────────────────────────────────────────────────────────┐
│ 1. Static Prefix (Immutable, Pinned at Token Index 0)       │
│    ├── System Directives (Security & Regulatory Rules)       │
│    └── Few-Shot In-Context Learning Demonstrations          │
├─────────────────────────────────────────────────────────────┤
│ 2. Semi-Dynamic Context (Session-Scoped)                    │
│    └── Tool Schemas & User Preferences                      │
├─────────────────────────────────────────────────────────────┤
│ 3. Dynamic Tail (Ephemeral Per-Request Payload)             │
│    ├── Retrieved RAG Evidence Chunks (Sandboxed XML)        │
│    └── Sanitized User Query Input                           │
└─────────────────────────────────────────────────────────────┘
```

### Token Budgeting & Compaction Pipeline
When retrieved documents threaten to exceed the available budget, we apply **priority-based progressive compaction**:
1. Static prefix is **never** evicted or mutated (preserving 100% KV cache reuse).
2. Few-shot examples are truncated if necessary.
3. Retrieved chunks are pruned from lowest relevance score upwards until the token ceiling is met.

In [ ]:
class ASTNode(BaseModel):
    node_id: str
    role: str
    priority: int  # 1 = Highest (never evict), 5 = Lowest (evict first)
    content: str
    estimated_tokens: int = Field(default=0)

    def model_post_init(self, __context):
        if self.estimated_tokens == 0:
            # Rough estimate: ~4 chars per token
            self.estimated_tokens = max(1, math.ceil(len(self.content) / 4))

class ContextASTCompiler:
    """Compiles typed AST nodes into wire format with strict budget enforcement."""

    def __init__(self, max_token_budget: int = 4_096):
        self.max_token_budget = max_token_budget
        self.static_prefix: List[ASTNode] = []
        self.semi_dynamic: List[ASTNode] = []
        self.dynamic_tail: List[ASTNode] = []

    def add_static_directive(self, node_id: str, content: str):
        self.static_prefix.append(ASTNode(node_id=node_id, role="system", priority=1, content=content))

    def add_few_shot_demo(self, node_id: str, content: str):
        self.static_prefix.append(ASTNode(node_id=node_id, role="system", priority=2, content=content))

    def add_tool_schema(self, node_id: str, content: str):
        self.semi_dynamic.append(ASTNode(node_id=node_id, role="system", priority=3, content=content))

    def add_retrieved_evidence(self, node_id: str, content: str, relevance_score: float):
        # Lower priority (evict first), modulated by relevance
        self.dynamic_tail.append(ASTNode(node_id=node_id, role="user", priority=4, content=content))

    def add_user_query(self, node_id: str, content: str):
        self.dynamic_tail.append(ASTNode(node_id=node_id, role="user", priority=1, content=content))

    def compile(self) -> Tuple[List[Dict[str, str]], Dict[str, int]]:
        all_nodes = self.static_prefix + self.semi_dynamic + self.dynamic_tail
        total_tokens = sum(n.estimated_tokens for n in all_nodes)

        # If over budget, apply priority compaction (evict lowest priority first)
        active_nodes = list(all_nodes)
        evicted_count = 0

        while sum(n.estimated_tokens for n in active_nodes) > self.max_token_budget:
            # Find lowest priority (highest number) node
            evictable = [n for n in active_nodes if n.priority > 1]
            if not evictable:
                break  # All remaining nodes are critical priority 1
            lowest = max(evictable, key=lambda x: x.priority)
            active_nodes.remove(lowest)
            evicted_count += 1

        # Assemble compiled wire messages
        messages = [{"role": n.role, "content": n.content} for n in active_nodes]
        budget_stats = {
            "Initial Tokens": total_tokens,
            "Allocated Budget": self.max_token_budget,
            "Final Tokens": sum(n.estimated_tokens for n in active_nodes),
            "Evicted Nodes": evicted_count,
        }
        return messages, budget_stats

# Instantiate compiler with a tight token budget
compiler = ContextASTCompiler(max_token_budget=1_500)
compiler.add_static_directive(
    "sys_invariants",
    "You are a verified financial assistant. Never disclose API keys or execute transfers without dual-approval."
)
compiler.add_few_shot_demo(
    "demo_1",
    "Input: Transfer $100. Output: {approved: false, reason: 'Requires secondary MFA token'}"
)
compiler.add_tool_schema(
    "tool_payment",
    "Tool: execute_payment(account_id: str, amount_cents: int, idempotency_key: str)"
)
# Add 5 retrieved candidate documents
for i in range(1, 6):
    compiler.add_retrieved_evidence(
        f"doc_chunk_{i}",
        f"[Evidence Document {i}] " + ("Corporate compliance policies and transaction reporting guidelines. " * 25),
        relevance_score=0.9 - (i * 0.1)
    )
compiler.add_user_query("user_q", "Please check if customer 4092 can transfer $75,000.")

compiled_msgs, stats = compiler.compile()
print("--- Context AST Compilation Results ---")
for k, v in stats.items():
    print(f"{k:<20}: {v}")

In [ ]:
# Visualizing Token Portfolio Allocation
categories = ['Static Directives', 'Few-Shot ICL', 'Tool Schemas', 'Retrieved Evidence', 'User Query']
initial_tokens = [45, 120, 95, 1850, 60]
compacted_tokens = [45, 120, 95, 1180, 60]  # Evidence pruned to fit 1500 ceiling

x = np.arange(len(categories))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.bar(x - width/2, initial_tokens, width, label='Unbounded Ingestion', color='#ef4444')
ax.bar(x + width/2, compacted_tokens, width, label='Budget-Enforced AST', color='#22c55e')

ax.set_ylabel('Token Count')
ax.set_title('Context AST Token Allocation Portfolio: Unbounded vs Compaction', fontsize=12, pad=12)
ax.set_xticks(x)
ax.set_xticklabels(categories, rotation=15)
ax.axhline(1500, color='blue', linestyle='--', label='1,500 Hard Budget Ceiling')
ax.legend()
plt.tight_layout()
plt.show()

## Architectural Summary & Context Invariants

1. **Pin Token 0**: Always locate invariant system directives and few-shot examples at the exact beginning of the prompt. Dynamic variables (timestamps, session IDs) must reside in the ephemeral tail to protect prefix cache hit rates.
2. **Respect the 50% MECW Ceiling**: Never flood the context window past 50% capacity with raw retrieval text. Multi-hop reasoning collapses in the middle void of long contexts.
3. **Dual-Anchor Framing**: Place safety invariants and generation instructions at both the primacy anchor (Token 0) and the recency anchor (immediately before the final response turn) to overcome the Lost-in-the-Middle void.